# 03 - Model Selection and Evaluation

Comparar runs y validar el modelo final en test.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import torch
import wandb
from sklearn.metrics import ConfusionMatrixDisplay

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from real_estate_ml.config import load_config
from real_estate_ml.constants import CLASSES
from real_estate_ml.data.dataset import get_dataloaders
from real_estate_ml.models.classifier import build_model
from real_estate_ml.training.engine import run_epoch

cfg = load_config(ROOT / "configs" / "base_config.yaml")
cfg

In [ ]:
api = wandb.Api()
project_path = f"{cfg['entity']}/{cfg['project_name']}"
runs = api.runs(project_path)

summary_rows = []
for run in runs:
    summary_rows.append({
        "name": run.name,
        "id": run.id,
        "state": run.state,
        "val_macro_f1": run.summary.get("val/macro_f1"),
        "test_macro_f1": run.summary.get("test/macro_f1"),
    })

import pandas as pd
runs_df = pd.DataFrame(summary_rows).sort_values("val_macro_f1", ascending=False)
runs_df.head(10)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
checkpoint_path = ROOT / "artifacts" / "best_model.pth"

model = build_model(
    backbone=cfg["model"]["backbone"],
    num_classes=cfg["data"]["num_classes"],
    pretrained=False,
    dropout=cfg["model"]["dropout"],
    freeze_backbone=False,
).to(device)
state = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(state["model_state_dict"])

loaders = get_dataloaders(
    data_dir=str(ROOT / cfg["data"]["data_dir"]),
    batch_size=cfg["data"]["batch_size"],
    image_size=cfg["data"]["image_size"],
    num_workers=cfg["data"]["num_workers"],
)

criterion = torch.nn.CrossEntropyLoss()
test_metrics = run_epoch(model, loaders["test"], criterion, optimizer=None, device=device, train=False)
print("Test macro_f1:", round(test_metrics.macro_f1, 4))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
ConfusionMatrixDisplay(test_metrics.confusion_matrix, display_labels=CLASSES).plot(
    xticks_rotation=45,
    ax=ax,
    colorbar=False,
)
plt.tight_layout()
plt.show()